In [1]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-ym_lycar
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-ym_lycar
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.0 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=f58f3935e1eb8212dac1cae30cf87e6da1feed43eed90a6b15206e19d3431d65
  Stored in directory: /tmp/pip-ephem-wheel-cache-ychrhwwu/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import copy
import json
import os
from sklearn.model_selection import train_test_split

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay
)
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from scipy.stats import spearmanr


ROOT_DIR = "/content/drive/MyDrive/CompNeuroscience-P1"
FEATURES_DIR = f"{ROOT_DIR}/lamem_features2"
SAVE_DIR = f"{ROOT_DIR}/memorability_models"
os.makedirs(SAVE_DIR, exist_ok=True)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Mounted at /content/drive
Device: cuda


In [3]:


import clip as openai_clip

ROOT_DIR     = "/content/drive/MyDrive/CompNeuroscience-P1"
FEATURES_DIR = f"{ROOT_DIR}/lamem_features2"
SAVE_DIR     = f"{ROOT_DIR}/memorability_models"
os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
EMOTIONS = ["amusement", "excitement", "awe", "contentment", "sadness"]
print(f"Device: {DEVICE}")

Device: cuda


In [4]:
df         = pd.read_parquet(f"{FEATURES_DIR}/lamem_features_full.parquet")
clip_mat   = np.load(f"{FEATURES_DIR}/clip_embeddings.npy")
dino_mat   = np.load(f"{FEATURES_DIR}/dino_embeddings.npy")
sbert_mat  = np.load(f"{FEATURES_DIR}/sbert_embeddings.npy")

memscore       = df["memscore"].values.astype(np.float32)
emotion_scores = df[[f"emotion_{e}" for e in EMOTIONS]].values.astype(np.float32)
valence_arousal = df[["valence", "arousal"]].values.astype(np.float32)

clip_norm  = clip_mat
dino_norm  = dino_mat
sbert_norm = sbert_mat
emo_norm   = emotion_scores
va_norm    = valence_arousal

all_idx    = np.arange(len(df))
strat_bins = pd.qcut(df["memscore"], q=5, labels=False, duplicates="drop")

train_idx, temp_idx = train_test_split(
    all_idx, test_size=0.3, random_state=42, stratify=strat_bins
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.4, random_state=42,
    stratify=strat_bins.iloc[temp_idx].values
)
print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

Train: 11770 | Val: 3027 | Test: 2018


In [5]:
class MemorabilityDataset(Dataset):
    def __init__(self, X, scores, indices):
        self.x = torch.tensor(X[indices],      dtype=torch.float32)
        self.y = torch.tensor(scores[indices], dtype=torch.float32)
    def __len__(self):          return len(self.y)
    def __getitem__(self, i):   return self.x[i], self.y[i]


def make_loaders(X, batch_size=512):
    return (
        DataLoader(MemorabilityDataset(X, memscore, train_idx),
                   batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True),
        DataLoader(MemorabilityDataset(X, memscore, val_idx),
                   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
        DataLoader(MemorabilityDataset(X, memscore, test_idx),
                   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
    )


class LinearProbe(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
    def forward(self, x):
        return self.linear(x).squeeze(-1)


def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in loader:
            preds.append(model(x.to(DEVICE)).cpu())
            targets.append(y)
    preds   = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()
    mse     = float(np.mean((preds - targets) ** 2))
    r, _    = pearsonr(preds, targets)
    rho, _  = spearmanr(preds, targets)
    return mse, float(r), float(rho)


def train_experiment(X, lr=1e-3, weight_decay=1e-4,
                     max_epochs=200, patience=15, batch_size=512, seed=42):
    torch.manual_seed(seed); np.random.seed(seed)
    train_loader, val_loader, test_loader = make_loaders(X, batch_size)
    model     = LinearProbe(X.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()
    best_val_mse  = float("inf")
    best_state    = None
    patience_left = patience
    history       = {"train_mse": [], "val_mse": [], "val_r": [], "val_rho": []}

    for epoch in range(1, max_epochs + 1):
        model.train()
        epoch_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(y)
        val_mse, val_r, val_rho = evaluate(model, val_loader)
        history["train_mse"].append(epoch_loss / len(train_loader.dataset))
        history["val_mse"].append(val_mse)
        history["val_r"].append(val_r)
        history["val_rho"].append(val_rho)
        if val_mse < best_val_mse - 1e-6:
            best_val_mse  = val_mse
            best_state    = copy.deepcopy(model.state_dict())
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left == 0:
                break

    model.load_state_dict(best_state)
    test_mse, test_r, test_rho = evaluate(model, test_loader)
    history["test_mse"] = test_mse
    history["test_r"]   = test_r
    history["test_rho"] = test_rho
    return model, history

In [6]:
clip_model, clip_preprocess = openai_clip.load("ViT-L/14", device=DEVICE)
clip_model = clip_model.float().eval()

with torch.no_grad():
    emotion_tokens   = openai_clip.tokenize(
        [f"a photo that conveys {e}" for e in EMOTIONS]
    ).to(DEVICE)
    emotion_text_emb = clip_model.encode_text(emotion_tokens).float()
    emotion_text_emb = emotion_text_emb / (emotion_text_emb.norm(dim=-1, keepdim=True) + 1e-8)

emotion_text_emb_np = emotion_text_emb.cpu().numpy()
print("emotion_text_emb:", emotion_text_emb_np.shape)

100%|███████████████████████████████████████| 890M/890M [00:13<00:00, 69.9MiB/s]


emotion_text_emb: (5, 768)


In [7]:
from PIL import Image
from tqdm import tqdm


def extract_patch_tokens(image_paths, batch_size=32, save_dir="checkpoints"):
    os.makedirs(save_dir, exist_ok=True)

    all_tokens = []

    for start in tqdm(range(0, len(image_paths), batch_size), desc="Extracting patch tokens"):
        batch_idx = start // batch_size
        save_path = os.path.join(save_dir, f"batch_{batch_idx}.npy")

        if os.path.exists(save_path):
            tokens = np.load(save_path)
            all_tokens.append(tokens)
            continue

        batch_paths = image_paths[start:start + batch_size]

        imgs = torch.stack([
            clip_preprocess(Image.open(p).convert("RGB"))
            for p in batch_paths
        ]).to(DEVICE).float()

        activations = {}

        def hook_fn(module, input, output):
            activations["tokens"] = output.detach()

        hook = clip_model.visual.ln_post.register_forward_hook(hook_fn)

        with torch.no_grad():
            clip_model.encode_image(imgs)

        hook.remove()

        tokens = activations["tokens"].cpu().numpy()

        np.save(save_path, tokens)

        all_tokens.append(tokens)

    return np.concatenate(all_tokens, axis=0)


def load_all_tokens(save_dir="checkpoints"):
    files = sorted(os.listdir(save_dir), key=lambda x: int(x.split("_")[1].split(".")[0]))
    all_tokens = [np.load(os.path.join(save_dir, f)) for f in files]
    return np.concatenate(all_tokens, axis=0)


In [ ]:
image_paths     = [f"{ROOT_DIR}/lamem_images/{name}" for name in df["name"].values]
patch_tokens_all = extract_patch_tokens(image_paths, batch_size=32, save_dir=FEATURES_DIR)
print("patch_tokens_all:", patch_tokens_all.shape)

Extracting patch tokens: 100%|██████████| 526/526 [1:45:32<00:00, 12.04s/it]

patch_tokens_all: (16815, 1024)


In [ ]:
patch_tokens_all = load_all_tokens("checkpoints")

print("patch_tokens_all:", patch_tokens_all.shape)


FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints'

In [ ]:
train_patches      = patch_tokens_all[train_idx]
train_patches_nocls = train_patches[:, 1:, :]
patches_flat       = train_patches_nocls.reshape(-1, train_patches_nocls.shape[-1])
patches_flat_norm  = patches_flat / (np.linalg.norm(patches_flat, axis=1, keepdims=True) + 1e-8)

M = patches_flat_norm.T @ emotion_text_emb_np.T
U, S, Vt = np.linalg.svd(M, full_matrices=False)

print(f"M: {M.shape} | U: {U.shape} | S: {S.round(3)} | Vt: {Vt.shape}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].bar(range(len(S)), S, color="steelblue", alpha=0.8)
axes[0].set_xticks(range(len(S)))
axes[0].set_xticklabels([f"Comp {i}" for i in range(len(S))])
axes[0].set_ylabel("Singular value")
axes[0].set_title("SVD spectrum (ViT × Emotion)")
axes[0].grid(axis="y", alpha=0.3)

im = axes[1].imshow(Vt, cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
axes[1].set_xticks(range(len(EMOTIONS)))
axes[1].set_xticklabels(EMOTIONS, rotation=30, ha="right")
axes[1].set_yticks(range(len(S)))
axes[1].set_yticklabels([f"Comp {i}" for i in range(len(S))])
axes[1].set_title("Emotion composition per SVD component (V^T)")
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/svd_spectrum_vt.png", dpi=150)
plt.show()

In [ ]:
def compute_alignment_features(patch_tokens, U, mode="max_and_mean"):
    patches_nocls = patch_tokens[:, 1:, :]
    proj = patches_nocls @ U
    if mode == "max":
        return proj.max(axis=1)
    elif mode == "mean":
        return proj.mean(axis=1)
    elif mode == "max_and_mean":
        return np.concatenate([proj.max(axis=1), proj.mean(axis=1)], axis=1)


k           = len(S)
align_feats = compute_alignment_features(patch_tokens_all, U[:, :k], mode="max_and_mean")
print("align_feats:", align_feats.shape)

In [ ]:
EXPERIMENTS = {
    "A_vit_only": {
        "blocks": [clip_norm],
        "desc":   "ViT only (baseline)",
    },
    "C_vit_emo": {
        "blocks": [clip_norm, emo_norm],
        "desc":   "ViT + emotion scores",
    },
    "D_vit_emo_va": {
        "blocks": [clip_norm, emo_norm, va_norm],
        "desc":   "ViT + emotion scores + valence/arousal",
    },
    "F_vit_full": {
        "blocks": [clip_norm, emo_norm, va_norm, sbert_norm],
        "desc":   "ViT + emotion scores + valence/arousal + SBERT",
    },
    "I_vit_align": {
        "blocks": [clip_norm, align_feats],
        "desc":   "ViT + SVD alignment features",
    },
    "J_align_only": {
        "blocks": [align_feats],
        "desc":   "SVD alignment features only",
    },
    "K_vit_emo_align": {
        "blocks": [clip_norm, emo_norm, align_feats],
        "desc":   "ViT + emotion scores + SVD alignment features",
    },
    "L_vit_full_align": {
        "blocks": [clip_norm, emo_norm, va_norm, sbert_norm, align_feats],
        "desc":   "ViT full + SVD alignment features",
    },
}

for name, cfg in EXPERIMENTS.items():
    cfg["X"]   = np.concatenate(cfg["blocks"], axis=1)
    cfg["dim"] = cfg["X"].shape[1]

print(f"{'Experiment':<22} {'dim':>6}   Description")
print("-" * 70)
for name, cfg in EXPERIMENTS.items():
    print(f"  {name:<20} {cfg['dim']:>6}   {cfg['desc']}")

In [ ]:
trained = {}

for name, cfg in EXPERIMENTS.items():
    model, history = train_experiment(cfg["X"])
    trained[name]  = (model, history)
    print(f"{name:<22}  test r={history['test_r']:.4f}  rho={history['test_rho']:.4f}")

In [ ]:
baseline_r   = trained["A_vit_only"][1]["test_r"]
baseline_rho = trained["A_vit_only"][1]["test_rho"]

rows = []
for name, cfg in EXPERIMENTS.items():
    _, hist = trained[name]
    rows.append({
        "Experiment" : name,
        "Description": cfg["desc"],
        "dim"        : cfg["dim"],
        "Test r"     : round(hist["test_r"],   4),
        "Test rho"   : round(hist["test_rho"],  4),
        "delta_r"    : round(hist["test_r"]   - baseline_r,   4),
        "delta_rho"  : round(hist["test_rho"] - baseline_rho, 4),
    })

results_df = pd.DataFrame(rows).sort_values("Test r", ascending=False)
print(results_df.to_string(index=False))
results_df.to_csv(f"{SAVE_DIR}/alignment_results.csv", index=False)

names  = results_df["Experiment"].tolist()
r_vals = results_df["Test r"].tolist()
colors = ["#2ecc71" if v >= baseline_r else "#e74c3c" for v in r_vals]

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(names, r_vals, color=colors, edgecolor="white", linewidth=0.8)
ax.axhline(baseline_r, color="black", linewidth=1.2, linestyle="--",
           label=f"Baseline (ViT only) r={baseline_r:.4f}")
ax.set_ylabel("Test Pearson r")
ax.set_title("Test Pearson r — ViT × Emotion alignment experiments")
ax.set_xticklabels(names, rotation=30, ha="right", fontsize=9)
ax.legend()
ax.grid(axis="y", alpha=0.3)
for bar, val in zip(bars, r_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.001,
            f"{val:.4f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/alignment_r_comparison.png", dpi=150)
plt.show()

In [ ]:
def plot_vit_emotion_alignment(image_path, patch_tokens_img, U, S, Vt,
                                top_k=3, patch_grid=14):
    patches = patch_tokens_img[1:]
    proj    = patches @ U[:, :top_k]

    fig, axes = plt.subplots(1, top_k + 1, figsize=(4 * (top_k + 1), 4))
    img     = Image.open(image_path).convert("RGB").resize((224, 224))
    img_arr = np.array(img).astype(np.float32) / 255.0

    axes[0].imshow(img)
    axes[0].set_title("Original", fontsize=9)
    axes[0].axis("off")

    for i in range(top_k):
        patch_map = proj[:, i].reshape(patch_grid, patch_grid)
        vmin = np.percentile(patch_map, 20)
        vmax = np.percentile(patch_map, 95)
        patch_norm = np.clip((patch_map - vmin) / (vmax - vmin + 1e-8), 0, 1)

        heat = np.array(
            Image.fromarray((patch_norm * 255).astype(np.uint8)).resize((224, 224))
        ) / 255.0

        overlay = np.zeros_like(img_arr)
        overlay[..., 0] = heat
        blended = np.clip(0.55 * overlay + 0.45 * img_arr, 0, 1)

        axes[i + 1].imshow(blended)

        emo_weights  = Vt[i]
        top_emo_idx  = np.argmax(np.abs(emo_weights))
        top_emo      = EMOTIONS[top_emo_idx]
        sign         = "+" if emo_weights[top_emo_idx] > 0 else "-"
        axes[i + 1].set_title(f"Comp {i}  (sigma={S[i]:.1f})\n{sign}{top_emo}", fontsize=9)
        axes[i + 1].axis("off")

    plt.tight_layout()
    return fig


def get_test_preds_errors(exp_name):
    model = trained[exp_name][0]
    X     = EXPERIMENTS[exp_name]["X"]
    _, _, test_loader = make_loaders(X)
    preds, targets = [], []
    model.eval()
    with torch.no_grad():
        for x, y in test_loader:
            preds.append(model(x.to(DEVICE)).cpu().numpy())
            targets.append(y.numpy())
    preds   = np.concatenate(preds)
    targets = np.concatenate(targets)
    errors  = np.abs(preds - targets)
    return preds, targets, errors


def plot_alignment_cases(exp_name="C_vit_emo", n_cases=3, mode="best", top_k=3):
    preds, targets, errors = get_test_preds_errors(exp_name)
    indices = np.argsort(errors)[:n_cases] if mode == "best" else np.argsort(errors)[-n_cases:][::-1]

    for i in indices:
        g_idx    = test_idx[i]
        img_path = f"{ROOT_DIR}/lamem_images/{df.iloc[g_idx]['name']}"
        dom_emo  = df.iloc[g_idx]["dominant_emotion"]
        fig      = plot_vit_emotion_alignment(
            img_path, patch_tokens_all[g_idx], U, S, Vt, top_k=top_k
        )
        fig.suptitle(
            f"{mode.upper()} | mem={targets[i]:.3f}  pred={preds[i]:.3f}  "
            f"err={errors[i]:.3f} | emotion={dom_emo}",
            fontsize=9, y=1.02
        )
        plt.show()

In [ ]:
plot_alignment_cases(exp_name="C_vit_emo", n_cases=3, mode="best",  top_k=3)
plot_alignment_cases(exp_name="C_vit_emo", n_cases=3, mode="worst", top_k=3)

In [ ]:
def plot_svd_top_images(U, S, Vt, patch_tokens_all, images_df, top_n=6,
                         top_k=3, patch_grid=14):
    patches_nocls = patch_tokens_all[:, 1:, :]
    proj_all      = patches_nocls @ U[:, :top_k]

    for comp in range(top_k):
        comp_scores  = proj_all[:, :, comp].max(axis=1)
        top_indices  = np.argsort(comp_scores)[-top_n:][::-1]

        emo_weights = Vt[comp]
        top_emo_idx = np.argmax(np.abs(emo_weights))
        top_emo     = EMOTIONS[top_emo_idx]
        sign        = "+" if emo_weights[top_emo_idx] > 0 else "-"

        fig, axes = plt.subplots(2, top_n, figsize=(top_n * 2.2, 5))

        for j, idx in enumerate(top_indices):
            img_path = f"{ROOT_DIR}/lamem_images/{images_df.iloc[idx]['name']}"
            img      = Image.open(img_path).convert("RGB").resize((224, 224))
            img_arr  = np.array(img).astype(np.float32) / 255.0

            patch_map  = proj_all[idx, :, comp].reshape(patch_grid, patch_grid)
            vmin       = np.percentile(patch_map, 20)
            vmax       = np.percentile(patch_map, 95)
            patch_norm = np.clip((patch_map - vmin) / (vmax - vmin + 1e-8), 0, 1)
            heat       = np.array(
                Image.fromarray((patch_norm * 255).astype(np.uint8)).resize((224, 224))
            ) / 255.0
            overlay        = np.zeros_like(img_arr)
            overlay[..., 0] = heat
            blended        = np.clip(0.55 * overlay + 0.45 * img_arr, 0, 1)

            axes[0, j].imshow(img)
            axes[0, j].axis("off")
            axes[0, j].set_title(f"mem={images_df.iloc[idx]['memscore']:.2f}", fontsize=7)

            axes[1, j].imshow(blended)
            axes[1, j].axis("off")

        fig.suptitle(
            f"Comp {comp}  (sigma={S[comp]:.2f})  —  dominant emotion: {sign}{top_emo}",
            fontsize=10
        )
        plt.tight_layout()
        plt.savefig(f"{SAVE_DIR}/svd_top_images_comp{comp}.png", dpi=150, bbox_inches="tight")
        plt.show()


plot_svd_top_images(U, S, Vt, patch_tokens_all, df, top_n=6, top_k=len(S))

In [ ]:
def cosine_sim_vit_emotion_per_image(patch_tokens_all, emotion_text_emb_np):
    patches_nocls = patch_tokens_all[:, 1:, :]
    patches_norm  = patches_nocls / (np.linalg.norm(patches_nocls, axis=-1, keepdims=True) + 1e-8)
    cos_sim       = patches_norm @ emotion_text_emb_np.T
    return cos_sim


cos_sim_all = cosine_sim_vit_emotion_per_image(patch_tokens_all, emotion_text_emb_np)
print("cos_sim_all:", cos_sim_all.shape)

fig, axes = plt.subplots(1, len(EMOTIONS), figsize=(14, 3), sharey=False)
for i, emo in enumerate(EMOTIONS):
    max_cos_per_image = cos_sim_all[:, :, i].max(axis=1)
    r, _ = pearsonr(max_cos_per_image, memscore)
    axes[i].scatter(max_cos_per_image, memscore, alpha=0.15, s=4, color="steelblue")
    axes[i].set_title(f"{emo}\nr={r:.3f}", fontsize=9)
    axes[i].set_xlabel("max patch cos sim", fontsize=8)
    axes[i].grid(alpha=0.3)
axes[0].set_ylabel("memorability", fontsize=8)
plt.suptitle("Max patch cosine similarity vs memorability per emotion", fontsize=10)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/cos_sim_vs_memscore.png", dpi=150)
plt.show()

In [ ]:
def compute_spatial_alignment_score(patch_tokens_all, emotion_text_emb_np, U):
    patches_nocls = patch_tokens_all[:, 1:, :]
    u_proj        = patches_nocls @ U
    e_proj        = emotion_text_emb_np @ U
    e_proj_norm   = e_proj / (np.linalg.norm(e_proj, axis=1, keepdims=True) + 1e-8)
    alignment     = u_proj @ e_proj_norm.T
    return alignment


spatial_align = compute_spatial_alignment_score(
    patch_tokens_all, emotion_text_emb_np, U
)
print("spatial_align:", spatial_align.shape)

spatial_align_feats = np.concatenate([
    spatial_align.max(axis=1),
    spatial_align.mean(axis=1),
], axis=1)

EXPERIMENTS["M_vit_spatial"] = {
    "blocks": [clip_norm, spatial_align_feats],
    "desc":   "ViT + spatial alignment score (SVD-projected)",
    "X":      np.concatenate([clip_norm, spatial_align_feats], axis=1),
}
EXPERIMENTS["M_vit_spatial"]["dim"] = EXPERIMENTS["M_vit_spatial"]["X"].shape[1]

model, history = train_experiment(EXPERIMENTS["M_vit_spatial"]["X"])
trained["M_vit_spatial"] = (model, history)
print(f"M_vit_spatial  test r={history['test_r']:.4f}  rho={history['test_rho']:.4f}")

In [ ]:
all_names = list(trained.keys())
all_r     = [trained[n][1]["test_r"]   for n in all_names]
all_rho   = [trained[n][1]["test_rho"] for n in all_names]

final_df = pd.DataFrame({
    "Experiment" : all_names,
    "Description": [EXPERIMENTS[n]["desc"] for n in all_names],
    "Test r"     : [round(r, 4) for r in all_r],
    "Test rho"   : [round(r, 4) for r in all_rho],
    "delta_r"    : [round(r - baseline_r, 4) for r in all_r],
}).sort_values("Test r", ascending=False)

print(final_df.to_string(index=False))
final_df.to_csv(f"{SAVE_DIR}/final_alignment_summary.csv", index=False)

for name, (model, _) in trained.items():
    torch.save(model.state_dict(), f"{SAVE_DIR}/model_{name}.pt")

np.save(f"{SAVE_DIR}/svd_U.npy", U)
np.save(f"{SAVE_DIR}/svd_S.npy", S)
np.save(f"{SAVE_DIR}/svd_Vt.npy", Vt)
np.save(f"{SAVE_DIR}/patch_tokens_all.npy", patch_tokens_all)
np.save(f"{SAVE_DIR}/align_feats.npy", align_feats)

print(f"\nAll models and arrays saved to: {SAVE_DIR}")